# Stage 4 — Small Reasoning Probe: Molmo2 vs Qwen3-VL vs InternVL3

Self-contained (no `git clone`/`src` import).

**What this tests:** not symbol detection — a short counting/comparison question over
the image ("which symbol type is most common here, and roughly how many?"), checked
against real ground-truth counts from the Kaggle labels. This is a small, indicative
signal about general reasoning/instruction-following, not a rigorous benchmark — one
prompt, a handful of tiles, judged by eye against the real answer.

**Why this test specifically:** Molmo2 is trained heavily around pointing-style output;
Qwen3-VL/InternVL3 are general-purpose VLMs more used to open-ended Q&A. If Molmo2
struggles to give a coherent counting/comparison answer while the other two manage it,
that's a real (if small) data point in the direction of the "unproven on reasoning"
caveat already in CLAUDE.md — not proof, an indication.

One model is loaded, tested, and freed before the next loads, to fit on one GPU.

## 1. Check GPU, install Kaggle CLI (no Google Drive needed)

Skips `drive.mount()` entirely — that's what's been triggering the account
verification-code friction. Downloads a handful of P&ID images directly from Kaggle's
own API into this runtime's local storage instead.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q kaggle

## 1b. Kaggle auth — paste your token here (not stored on Drive)

Get a token from kaggle.com → Settings → API → Create New Token, then paste just the
token string below. This is a Kaggle account credential, unrelated to the Google/Drive
auth that's been causing friction.

In [ ]:
import os

KAGGLE_TOKEN = "paste-your-kaggle-token-here"  # <-- edit this

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
token_path = os.path.expanduser("~/.kaggle/access_token")
with open(token_path, "w") as f:
    f.write(KAGGLE_TOKEN)
os.chmod(token_path, 0o600)
print("Kaggle token installed for this session.")

## 1c. Download a small sample of the P&ID dataset (not the full 30k images)

In [ ]:
KAGGLE_LOCAL_DIR = "/content/kaggle_pid_symbols"

!kaggle datasets download -d hristohristov21/pid-symbols -p /content/kaggle_dl --unzip
!mkdir -p {KAGGLE_LOCAL_DIR}
!mv /content/kaggle_dl/* {KAGGLE_LOCAL_DIR}/ 2>/dev/null || true
print("Downloaded to", KAGGLE_LOCAL_DIR)

## 2. Pick 3 labeled Kaggle tiles + compute real ground-truth majority class

(No model loaded yet — just picking tiles and reading their real labels, so we know
the correct answer before asking anything.)

In [ ]:
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image

kaggle_p = Path(KAGGLE_LOCAL_DIR)

# No classes.json available without Drive — using the same names documented in
# Stage4_Phase2_Data_Preparation.ipynb (class 0 confirmed unused, verified against
# real data: class 0 has 0 instances, class 21 has ~3x every other class).
class_names = {
    "0": "Not_used", "1": "Gate_Valve", "2": "Ball_Valve", "3": "Globe_valve_NO",
    "4": "Gate_valve_NO", "5": "Globe_valve_NO", "6": "Butterfly_valve", "7": "Plug_valve",
    "8": "Check_valve", "9": "Diaphragm_valve", "10": "Needle_valve",
    "11": "Half_Filled_Gate_Valve", "12": "Gate_Valve_NC", "13": "Globle_valve_NC",
    "14": "Control_Valve", "15": "Rotary_Valve", "16": "Ball_valve_NC", "17": "Paddle_blind",
    "18": "Spectacle_blind_Closed", "19": "Spectacle_blind_Open", "20": "Reducer",
    "21": "Flange_or_Nozzle", "22": "Rupture_disk", "23": "Pipe_Insulation_or_Tracing",
    "24": "Flow_Arrow", "25": "sight_glass", "26": "Instrument_Field", "27": "Instrument_Field",
    "28": "Instrument_Panel", "29": "Instrument_Aux_Panel", "30": "box",
    "31": "Instrument_Panel", "32": "box",
}

random.seed(1)
candidates = []
for lbl in (kaggle_p / "labels").glob("*.txt"):
    lines = [l for l in lbl.read_text().splitlines() if l.strip()]
    if len(lines) < 6:
        continue
    counts = Counter(l.split()[0] for l in lines)
    top_cls, top_n = counts.most_common(1)[0]
    # only keep tiles with an unambiguous majority (not a 3-way tie etc.)
    if top_n >= 3 and (len(counts) == 1 or top_n > counts.most_common(2)[1][1]):
        candidates.append((lbl, top_cls, top_n, len(lines)))
    if len(candidates) >= 200:
        break

probe_tiles = random.sample(candidates, 3)
for lbl, top_cls, top_n, total in probe_tiles:
    name = class_names.get(top_cls, f"class_{top_cls}")
    print(f"{lbl.stem:20s} majority={name:20s} count={top_n}  (total boxes in tile: {total})")

## 3. The reasoning prompt (same question, all 3 candidates)

In [ ]:
REASONING_PROMPT = (
    "Look at this P&ID tile. Which type of symbol appears most frequently in this image, "
    "and approximately how many of that symbol are there? Answer in one or two sentences."
)

def show_probe_images():
    for lbl, top_cls, top_n, total in probe_tiles:
        img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
        print(f"--- {lbl.stem} --- ground truth: {class_names.get(top_cls, top_cls)} x{top_n} (of {total} total boxes)")
        display(Image.open(img_path))

## 4. Qwen3-VL

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
qwen_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", dtype=torch.bfloat16, device_map="cuda"
)
print("Qwen3-VL loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": REASONING_PROMPT}]}]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(qwen_model.device)
    out = qwen_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = qwen_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("Qwen3-VL:", answer.strip())

del qwen_model, qwen_processor
torch.cuda.empty_cache()
print("\nFreed Qwen3-VL from GPU.")

## 5. InternVL3

In [ ]:
internvl_processor = AutoProcessor.from_pretrained("OpenGVLab/InternVL3-8B-hf")
internvl_model = AutoModelForImageTextToText.from_pretrained(
    "OpenGVLab/InternVL3-8B-hf", dtype=torch.bfloat16, device_map="cuda"
)
print("InternVL3 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": REASONING_PROMPT}]}]
    inputs = internvl_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(internvl_model.device)
    out = internvl_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = internvl_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("InternVL3:", answer.strip())

del internvl_model, internvl_processor
torch.cuda.empty_cache()
print("\nFreed InternVL3 from GPU.")

## 6. Molmo2-O-7B

Requires `transformers==4.57.1` — **restart the runtime after this install cell**, then
re-run cells 2-3 (tile selection) before continuing to the load+probe cell below.

In [ ]:
!pip install -q transformers==4.57.1 accelerate

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

molmo_processor = AutoProcessor.from_pretrained("allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto")
molmo_model = AutoModelForImageTextToText.from_pretrained(
    "allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto", device_map="cuda"
)
print("Molmo2 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "text", "text": REASONING_PROMPT}, {"type": "image", "image": img}]}]
    inputs = molmo_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(molmo_model.device)
    out = molmo_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = molmo_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("Molmo2:", answer.strip())

## 7. What to look for

For each tile, compare the three answers against the printed ground truth:
- Does the answer name a plausible symbol type, or is it garbled/off-topic/just points?
- Is the count in the right ballpark (doesn't need to be exact)?
- Does it actually answer the comparison question, or does it just describe the image generically?

Paste the three candidates' raw answers back — this determines what (if anything) goes
into the write-up. A model doing noticeably worse here isn't proof of anything on its own,
but it's a real, checkable data point, not an assumption.